In [12]:
import pandas as pd
import re
import requests
from bs4 import BeautifulSoup
import time, json


In [13]:
base_url = "https://www.pokernews.com/tours/wsop/2026-wsop/2026-wsop-main-event/"
day9  = scrape_day(15923, base_url + "day9/")
day10 = scrape_day(15924, base_url + "day10/")
day11 = scrape_day(15925, base_url + "day11/")

In [14]:
def extract_hands(html):
    soup = BeautifulSoup(html, "html.parser")
    out = []
    for post in soup.select("main.event__post__body") or [soup]:
        current = None
        for p in post.find_all("p"):
            if p.find("sub") or "event__tags" in (p.get("class") or []):
                continue
            text = p.get_text(" ", strip=True)
            m = re.match(r"Hand #(\d+):\s*(.*)", text)
            if m:
                current = {"hand": int(m.group(1)), "description": m.group(2)}
                out.append(current)
            elif current is not None:
                current["description"] += " " + text
    return out

In [15]:
def scrape_day(day_id, page_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    rows = {}
    anchor = False
    seen = set()

    for _ in range(50):
        u = ("https://www.pokernews.com/reporting-uroborus.json.php"
             f"?v2&day_id={day_id}&event_id=5587&filter=%5B%5D"
             f"&ts_anchor={anchor}&offset=10&page=0")
        r = requests.get(u, headers=headers)
        r.raise_for_status()
        j = r.json()

        payload = j["data"]
        html = payload if isinstance(payload, str) else json.dumps(payload)
        html = html.encode().decode("unicode_escape") if "\\u" in html else html
        for h in extract_hands(html):
            rows[h["hand"]] = h

        nxt = j["ts_anchor"]
        if not nxt or nxt in seen:
            break
        seen.add(nxt)
        anchor = nxt
        time.sleep(1)

    page = requests.get(page_url, headers=headers)
    page.raise_for_status()
    for h in extract_hands(page.text):
        rows[h["hand"]] = h

    return pd.DataFrame(rows.values()).sort_values("hand").reset_index(drop=True)

In [16]:
all_hands = (pd.concat([day9, day10, day11])
               .drop_duplicates(subset="hand")
               .sort_values("hand")
               .reset_index(drop=True))
all_hands.to_csv("all_hands.csv", index=False)


In [17]:
import re
import pandas as pd

PLAYERS = {
    "Jumalon": "Lucas Jumalon",
    "Saaskilahti": "Lauri Saaskilahti",
    "Mueller": "Greg Mueller",
    "Feng": "Han Feng",
    "Hammoud": "Rami Hammoud",
    "Gagliano": "Michael Gagliano",
    "Boos": "Mario Boos",
    "Evagorou": "Evagoras Evagorou",
    "Shaevel": "Jamie Shaevel",
}
NAME = "|".join(PLAYERS)
NAME_RE = re.compile(rf"\b({NAME})\b")

CARD_RE = re.compile(r"(?<![A-Za-z0-9])(10|[2-9TJQKA])\s*([♠♥♦♣])")
SUITS = {"♠": "s", "♥": "h", "♦": "d", "♣": "c"}
TOKEN_RE = re.compile(r"<([2-9TJQKA][shdc])>")
STREET_RE = re.compile(r"\b(flop|turn|river)\b", re.I)
POS_RE = re.compile(
    r"(under the gun|early position|middle position|hijack|cutoff|button|small blind|big blind)"
)
AMOUNT_RE = re.compile(r"\d{1,3}(?:,\d{3})+|\d{4,}")
IMPLIED_RE = re.compile(
    r"(?<!-)\b(?:it |the action |action )?folded (?:around )?to\b"
    r"|\bthe (?:rest of the|entire|whole) table folded\b"
    r"|\beveryone (?:else )?folded\b|\beverybody folded\b"
    r"|(?<!-)\bfolded around\b",
    re.I,
)
OTHERS_FOLD_RE = re.compile(
    r"\bthe (?:rest of the|entire|whole) table folded\b|\beveryone (?:else )?folded\b|\beverybody folded\b",
    re.I,
)

# Checked in this order; the first pattern that matches the clause wins.
VERBS = [
    ("walk", r"\b(?:received|receiving|got|gets?|getting) an? walk\b"),
    ("checkfold", r"\bcheck-fold(?:ed|s)?\b"),
    ("raise", r"\b(?:three-bet|3-bet|four-bet|4-bet|raised|raising|raise|re-raised|reraised|opened|raises)\b"),
    ("allin", r"\b(?:jammed|shoved|shoving|moved all[- ]in|all[- ]in|shoves|jams)\b"),
    ("bet", r"\b(?:continued|bet|bets)\b"),
    ("call", r"\b(?:called|calls|call|caller|defended|limped)\b"),
    ("check", r"\b(?:checked|checks|check)\b"),
    ("fold", r"\b(?:folded|folds|fold|mucked|laid down)\b|\blet (?:<[^>]+>\s*)+go\b"),
]
VERBS = [(n, re.compile(p, re.I)) for n, p in VERBS]


def norm_cards(text):
    def sub(m):
        rank = "T" if m.group(1) == "10" else m.group(1)
        return f"<{rank}{SUITS[m.group(2)]}>"
    return CARD_RE.sub(sub, text)


def clean(text):
    text = re.split(r"\bTags:", text)[0]               # drop trailing tag list
    text = re.sub(r'[“"][^”"]*[”"]', " ", text)        # drop quoted speech
    text = re.sub(r"\[[^\]]*\]", " ", text)             # drop [bracketed] asides
    return norm_cards(text)


def parse_hand(hand_no, text):
    text = clean(str(text))
    actions, hole, pos, notes = [], {}, {}, []
    info, pending = {}, []
    if re.search(r"not shown on (?:the )?broadcast", text, re.I):
        return actions, hole, pos, ["hand not shown on broadcast"], info
    lm = re.search(r"blinds? (?:went|moved|have gone|increased) up.*?to ([\d,]+)\s*/\s*([\d,]+)",
                   text, re.I)
    if lm:
        info["small_blind"] = int(lm.group(1).strip(",").replace(",", ""))
        info["big_blind"] = int(lm.group(2).strip(",").replace(",", ""))
        am = re.search(r"([\d,]{5,})\s+(?:big blind )?ante", text, re.I)
        if am:
            info["ante"] = int(am.group(1).strip(",").replace(",", ""))
    street, board = "preflop", []
    last_player = None
    last_aggressor = None

    def add(player, action, amount=None, note=""):
        actions.append({
            "hand": hand_no, "step": len(actions) + 1, "street": street,
            "player": player, "action": action, "amount": amount,
            "board": " ".join(board), "note": note,
        })
        if player is None:                  # fill from the next player mentioned
            pending.append(len(actions) - 1)

    for sent in re.split(r"(?<=[.!?])\s+", text):
        sent = re.sub(rf"\b({NAME}) and (?:\w+ )?({NAME})\b", r"\1 & \2", sent)
        for clause in re.split(r",\s+(?=[A-Za-z<])|\s+and\s+|\s+before\s+|\s+but\s+", sent):
            clause = clause.strip(" .")
            if not clause:
                continue

            cards = TOKEN_RE.findall(clause)
            sm = STREET_RE.search(clause)
            if sm and cards:                            # board cards
                street = sm.group(1).lower()
                board = cards[:3] if street == "flop" else board + cards[:1]

            implied = IMPLIED_RE.search(clause)
            body = IMPLIED_RE.sub(" ", clause) if implied else clause

            names = list(dict.fromkeys(PLAYERS[n] for n in NAME_RE.findall(body)))
            by = re.search(rf"\b(?:from|by)\s+(?:\w+\s+){{0,2}}({NAME})\b", body)
            if names and re.search(r"\bboth\b|&", body):
                actors = names
            elif by:
                actors = [PLAYERS[by.group(1)]]
            elif names:
                actors = [names[0]]
            elif re.search(r"\bboth players\b", body):
                actors = ["(both)"]
            else:
                actors = [last_player] if last_player else []

            if implied:
                if OTHERS_FOLD_RE.search(clause):
                    add(last_player, "all_others_fold",
                        note="everyone else folded (implied; needs seat order)")
                else:
                    target = names[0] if names else last_player
                    add(target, "folds_to",
                        note="players before this one folded (implied; needs seat order)")

            if actors and actors[0] != "(both)":
                pm = POS_RE.search(body)
                if pm:
                    pos.setdefault(actors[0], pm.group(1))
                if cards and not sm:
                    hole[actors[0]] = " ".join(cards[:2])

            verb = next((n for n, p in VERBS if p.search(body)), None)
            if verb:
                amt = AMOUNT_RE.search(body)
                amount = int(amt.group().replace(",", "")) if amt else None
                if not actors:
                    notes.append(f"no actor for: {clause}")
                for a in actors:
                    if verb == "checkfold":
                        add(a, "check")
                        if last_aggressor and last_aggressor != a:
                            add(last_aggressor, "bet", amount,
                                "inferred from 'check-folded to another bet'")
                        add(a, "fold")
                        notes.append("check-fold: opponent bet inferred")
                    else:
                        note = ""
                        if a == "(both)":
                            note = "both remaining players; expand using seat order"
                            notes.append("'both players' action needs expanding")
                        elif verb == "allin" and amount is None:
                            note = "all-in amount not stated; use starting stack"
                            notes.append("all-in without amount")
                        add(a, verb, amount, note)
                        if verb in ("raise", "bet", "allin"):
                            last_aggressor = a

            if actors and actors[0] != "(both)":
                last_player = actors[0]
                for i in pending:
                    actions[i]["player"] = last_player
                    actions[i]["note"] += " [player taken from next mention]"
                pending.clear()

    return actions, hole, pos, notes, info


def parse_all(df):
    acts, players, flags, levels = [], [], [], []
    for _, row in df.iterrows():
        h = int(row["hand"])
        a, hole, pos, notes, info = parse_hand(h, row["description"])
        acts += a
        if info:
            levels.append({"hand": h, **info})
        for p in dict.fromkeys(list(hole) + list(pos)):
            players.append({"hand": h, "player": p,
                            "hole_cards": hole.get(p, ""), "position": pos.get(p, "")})
        if notes or not a:
            flags.append({"hand": h, "notes": "; ".join(notes) or "no actions parsed"})
    actions_df = pd.DataFrame(acts)
    actions_df["amount"] = actions_df["amount"].astype("Int64")
    return actions_df, pd.DataFrame(players), pd.DataFrame(flags), pd.DataFrame(levels)

In [19]:
all_hands = pd.read_csv("all_hands.csv")
actions, players, flags, levels = parse_all(all_hands)

actions.to_csv("actions.csv", index=False)
players.to_csv("players.csv", index=False)
flags.to_csv("needs_review.csv", index=False)
levels.to_csv("levels.csv", index=False)
print(len(actions), len(players), len(flags), len(levels))

552 264 21 1
